# The Text Only Model

This model cannot be feed images or vision data, just text.

![text only model](../showcase_images/text-only-model.png)

In [1]:
import easyjupyter
import torch.nn as nn
import torch
from model.decoder import Decoder
from model.generator import Generator
from model.RoPE import precompute_freqs_cis
from typing import Optional

In [ ]:
from typing import TYPE_CHECKING
if TYPE_CHECKING:
    from configs import BaseConfig

In [3]:
class TextOnlyModel(nn.Module):
    def __init__(self, cfg: BaseConfig):
        """
        The text-only model. This model cannot be feed images or audio, only text.
        """
        super().__init__()
        self.cfg = cfg
        self.decoder = Decoder(cfg).to(cfg.device)

        self.token_embedding = nn.Embedding(
            num_embeddings=cfg.vocab_size, embedding_dim=cfg.d_model
        )

        self.generator = Generator(cfg)

        # Precompute RoPE frequencies
        self.freqs_cis = None
        self.compute_RoPE()
        self.initialize_weights()

    def compute_RoPE(self):
        """
        Compute RoPE representations. Once at the initialization, then again whenever the size of the context length changes, e.g., after the initial stage of pre-training the context length is increased.
        """
        self.freqs_cis = precompute_freqs_cis(
            cfg=self.cfg,
            head_dim=self.cfg.d_model // self.cfg.attn_heads,
            theta=self.cfg.pos_embeddings_freq,
        ).to(self.cfg.device)

    def initialize_weights(self):
        """Initialize parameters with Xavier uniform"""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

        print(
            f"\n\nModel initialized with {sum(p.numel() for p in self.parameters()):,} parameters!\n\n"
        )

    def forward(
        self,
        x,
        combined_mask: torch.Tensor,
        start_pos: int = 0,
        kv_cache: Optional[list[tuple]] = None,
    ):
        """
        Feed the model.
        Args:
            x: The input tensor of shape (batch_size, context_len).
            combined_mask: This mask contains the causal mask which prevents tokens from attending to future tokens, and the document mask which prevents a token in Doc A from attending to tokens in other documents.
                - Paper mentioned: "We use an attention mask that prevents self-attention between different documents within the same sequence. We find that this change had limited impact during in standard pre-training, but find it to be important in continued pre-training on very long sequences."
            start_pos: Handles the slicing of the documents for both training and inference. During training, start_pos is always 0, because we process the full context window at once. During inference, start_pos increments by 1 for each new token the model generates.
            kv_caches: Inference only, a list of kv caches.
        
        Returns:
            logits, updated_kv_cache
        """
        # Embed the input tokens
        x_embeds = self.token_embedding(x)

        # Slice the precomputed RoPE frequencies to match the context length, just in case the dataloader yields a smaller batch.
        context_len = x.shape[1]
        batch_freqs_cis = self.freqs_cis[start_pos:start_pos+context_len]

        # Run the Decoder
        decoder_out, updated_kv_cache = self.decoder(
            x_embeds, batch_freqs_cis, combined_mask, kv_cache
        )

        # Run the generator
        logits = self.generator(decoder_out)
        return logits, updated_kv_cache

In [4]:
# @i-c
# TEST
from llama_configs import Llama3_scaled_down
from model.utils.masking import create_causal_doc_mask

d_model = 512
vocab_size = 32000
context_len = 256
batch_size = 8
cfg = Llama3_scaled_down()

dummy_input = torch.randint(0, vocab_size, (batch_size, context_len), device=cfg.device)
mask = create_causal_doc_mask(cfg, dummy_input)


model = TextOnlyModel(cfg).to(cfg.device)
model(dummy_input, mask)


Project Root: /Users/tonyavis/Main/Build_an_LLM


Model initialized with 99,528,704 parameters!




(tensor([[[-0.2831,  0.2707,  0.4913,  ...,  0.0450, -0.2125,  0.1028],
          [-0.3307,  0.1982,  0.4201,  ...,  0.1176, -0.1205, -0.0227],
          [-0.3874,  0.1149,  0.4320,  ...,  0.1746, -0.0989,  0.0706],
          ...,
          [ 0.1235, -0.4529,  0.0278,  ...,  0.3445, -0.0878,  0.0851],
          [-0.0245, -0.4355,  0.1103,  ...,  0.4073,  0.0043, -0.0073],
          [ 0.1292, -0.5257,  0.1886,  ...,  0.2754, -0.0947,  0.2023]],
 
         [[-0.0678,  0.0447, -0.0042,  ...,  0.1773, -0.3020,  0.4026],
          [-0.1108,  0.1395,  0.0257,  ...,  0.1832, -0.2919,  0.3259],
          [-0.2138,  0.1864,  0.1292,  ...,  0.0738, -0.3580,  0.3276],
          ...,
          [ 0.5312, -0.0225, -0.1439,  ...,  0.1668,  0.0132, -0.3163],
          [ 0.4502,  0.0155, -0.0901,  ...,  0.2392, -0.1278, -0.1855],
          [ 0.5141,  0.0914, -0.1066,  ...,  0.1498, -0.1301, -0.2646]],
 
         [[ 0.1508, -0.0050,  0.0967,  ..., -0.3846,  0.0882, -0.0653],
          [ 0.2664, -0.1998,